In [1]:
# Imports and Configurations
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import time

# Clustering and distance metrics
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score

# Parallelization
import multiprocessing as mp

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

# Configuration
BENCHMARK      = '^GSPC'
RISK_FREE_RATE = 0.045
START_DATE     = '2005-01-01'
END_DATE       = '2026-01-01'
ROLLING_WINDOW = 52
MIN_HIST       = 400
K_VALUES       = [2, 4, 7]
N_CORES        = 14

# Three-period out-of-sample validation framework (20-year-window)
TRAIN_START = '2005-01-01'
TRAIN_END   = '2010-12-31'   # 6 years
VAL_START   = '2011-01-01'
VAL_END     = '2017-12-31'   # 7 years
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'   # 7 years

# Strategy Parameters (tightened version of Notebook 6)
ZSCORE_WINDOW   = 52
VOL_WINDOW      = 26
VOL_THRESHOLD   = 1.5
HIGH_VOL_ENTRY  = 2.25  # was 2.5
LOW_VOL_ENTRY   = 1.25  # was 1.5
STANDARD_ENTRY  = 1.75  # was 2.0
EXIT_THRESHOLD  = 0.5
STOP_LOSS       = 3.25  # was 3.5

print('Notebook 7: DTW vs OCP vs TOP Clustering - S&P 500, 20-Year Three-Period Framework')
print(f'Train:  {TRAIN_START} to {TRAIN_END} (6 years)')
print(f'Val:    {VAL_START} to {VAL_END} (7 years)')
print(f'Test:   {TEST_START} to {TEST_END} (7 years)')

Notebook 7: DTW vs OCP vs TOP Clustering - S&P 500, 20-Year Three-Period Framework
Train:  2005-01-01 to 2010-12-31 (6 years)
Val:    2011-01-01 to 2017-12-31 (7 years)
Test:   2018-01-01 to 2025-12-31 (7 years)


In [4]:
import re

# Retrieving S&P 500 constitutents as of January 1, 2005

# Survivorship-bias free source
hist_url = (
    'https://raw.githubusercontent.com/fja05680/sp500/master/'
    'S%26P%20500%20Historical%20Components%20%26%20Changes.csv'
)

hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])

# Targeting the row closest to our date
target_date = pd.Timestamp('2005-01-01')
snapshot_row = hist[hist['date'] <= target_date].sort_values('date').iloc[-1]

print(f'Using snapshot date: {snapshot_row["date"].date()}')

sp500_tickers_2005 = sorted(snapshot_row['tickers'].split(','))

raw_tickers = sorted(snapshot_row['tickers'].split(','))

# Stipping the dataset's own "-YYYYMM" delisting_date suffix convention
suffix_pattern = re.compile(r'-\d{6}$')

sp500_tickers_2005 = sorted(set(
    suffix_pattern.sub('', t.strip()).replace('.','-')
    for t in raw_tickers
))

n_suffixed = sum(1 for t in raw_tickers if suffix_pattern.search(t.strip()))
print(f'Tickers with a delisting-date suffix in the source data: {n_suffixed}')
print(f'S&P 500 constituents as of {target_date.date()}: {len(sp500_tickers_2005)} stocks')
print(f'\nFirst 10 tickers: {sp500_tickers_2005[:10]}')

Using snapshot date: 2004-12-30
Tickers with a delisting-date suffix in the source data: 179
S&P 500 constituents as of 2005-01-01: 495 stocks

First 10 tickers: ['A', 'AABA', 'AAPL', 'ABC', 'ABI', 'ABKFQ', 'ABS', 'ABT', 'ACS', 'ACV']


In [6]:
# Downloading price data for this S&P 500 205 universe
import yfinance as yf

print(f'Downloading price data for {len(sp500_tickers_2005)} stocks...')
print(f'Period: {START_DATE} to {END_DATE}, weekly (Wednesday), adjusted close\n')

raw = yf.download(
    sp500_tickers_2005,
    start=START_DATE,
    end=END_DATE,
    interval='1wk',
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')

# Dropping any tickers without complete history across the full history
before_drop = price_data.shape[1]
price_data  = price_data.ffill().dropna(axis=1)
after_drop  = price_data.shape[1]

# Dropping any tickers that are flatlined through the end of the series
TAIL_WINDOW = 10

flatlined = []
for t in price_data.columns:
    tail = price_data[t].iloc[-TAIL_WINDOW:]
    if tail.nunique() == t:
        flatlined.append(t)

price_data = price_data.drop(columns=flatlined)
after_drop = price_data.shape[1]

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers with complete history: {after_drop} (dropped {before_drop - after_drop} incomplete/delisted)')

dropped_tickers = sorted(set(sp500_tickers_2005) - set(price_data.columns))
print(f'\nDropped tickers ({len(dropped_tickers)}): {dropped_tickers}')
        

Period: 2005-01-01 to 2026-01-01, weekly (Wednesday), adjusted close



$HNZ: possibly delisted; no price data found  (1wk 2005-01-01 -> 2026-01-01)
$LEHMQ: possibly delisted; no price data found  (1wk 2005-01-01 -> 2026-01-01)
$FSL: possibly delisted; no price data found  (1wk 2005-01-01 -> 2026-01-01)
$SYMC: possibly delisted; no timezone found      ]  5 of 495 completed
[                       1%                       ]  7 of 495 completed$AVP: possibly delisted; no timezone found
$DPHIQ: possibly delisted; no timezone found     ]  11 of 495 completed
[*                      2%                       ]  12 of 495 completed$CTL: possibly delisted; no timezone found
$SPLS: possibly delisted; no price data found  (1wk 2005-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 1104555600, endDate = 1767243600")
[**                     4%                       ]  22 of 495 completed$HSH: possibly delisted; no price data found  (1wk 2005-01-01 -> 2026-01-01)
$MXIM: possibly delisted; no timezone found      ]  36 of 495 completed
$BRCM: possib


Price data shape: (1096, 267)
Date range: 2005-01-05 to 2025-12-31
Tickers with complete history: 267 (dropped 228 incomplete/delisted)

Dropped tickers (228): ['AABA', 'ABC', 'ABI', 'ABKFQ', 'ABS', 'ACS', 'ACV', 'ADCT', 'AGN', 'ALTR', 'AMCC', 'ANDW', 'ANTM', 'APC', 'APCC', 'APOL', 'ARNC', 'ASN', 'ASO', 'AT', 'AV', 'AVP', 'AW', 'AYE', 'BCR', 'BDK', 'BEAM', 'BHGE', 'BIG', 'BJS', 'BK', 'BLL', 'BLS', 'BMC', 'BMET', 'BMS', 'BNI', 'BOL', 'BR', 'BRCM', 'BSC', 'BUD', 'CA', 'CBE', 'CBS', 'CBSS', 'CCE', 'CCTYQ', 'CEG', 'CFC', 'CHIR', 'CIN', 'CITGQ', 'CMA', 'CMVT', 'CMX', 'COL', 'CPNLQ', 'CPWR', 'CTB', 'CTL', 'CTX', 'CTXS', 'DALRQ', 'DCNAQ', 'DELL', 'DG', 'DJ', 'DOW', 'DPHIQ', 'DYN', 'EC', 'EDS', 'EKDKQ', 'EMC', 'EOP', 'ESRX', 'ETFC', 'FDC', 'FDO', 'FII', 'FOXA', 'FRX', 'FSH', 'FSL', 'FTR', 'G', 'GAS', 'GDT', 'GDW', 'GENZ', 'GLK', 'GP', 'GPS', 'GR', 'GTW', 'HCA', 'HCR', 'HES', 'HET', 'HLT', 'HMA', 'HNZ', 'HOT', 'HPC', 'HSH', 'HSP', 'IGT', 'IPG', 'IR', 'JAVA', 'JCP', 'JNS', 'JNY', 'JP', 'JWN', '

In [7]:
print("""
    Data source limitation note (for methodology section / paper draft):

    yfinance does not serve historical price data for tickers that have 
    been delisted, regardless of how far in the past the requested date 
    range is. This means our S&P 500 2005 universe is effectively: '
    constituents as of Jan 2005 whose historical data remains accessible 
    via Yahoo Finance as of the analysis date' (2026), not strictly 
    'constituents as of Jan 2005.' Companies that were acquired, taken 
    private, or delisted at any point between 2005 and today are absent
    even thought they had complete price history during the actual 
    2005-2025 study window. This effect is matierally more substantial 
    here than in a shorter window, since a 20-year span captures far more 
    corporate turnover.
""")

print("""
    This is a known constraint of free retail data verndors and differs
    from the CRSP/WRDS standard in academic finance, which preserves
    full historical records for delisted securities. Disclosed here as a 
    limitation rather than corrected, due to project resources.
""")


    Data source limitation note (for methodology section / paper draft):

    yfinance does not serve historical price data for tickers that have 
    been delisted, regardless of how far in the past the requested date 
    range is. This means our S&P 500 2005 universe is effectively: '
    constituents as of Jan 2005 whose historical data remains accessible 
    via Yahoo Finance as of the analysis date' (2026), not strictly 
    'constituents as of Jan 2005.' Companies that were acquired, taken 
    private, or delisted at any point between 2005 and today are absent
    even thought they had complete price history during the actual 
    2005-2025 study window. This effect is matierally more substantial 
    here than in a shorter window, since a 20-year span captures far more 
    corporate turnover.


    This is a known constraint of free retail data verndors and differs
    from the CRSP/WRDS standard in academic finance, which preserves
    full historical records for delisted 

In [8]:
# Computing weekly log returns
log_returns = np.log(price_data / price_data.shift(1))

# Computing rolling sharpe ratio (risk-free-rate adjusted)
weekly_rf = RISK_FREE_RATE / 52

rolling_mean = log_returns.rolling(window=ROLLING_WINDOW).mean()
rolling_std = log_returns.rolling(window=ROLLING_WINDOW).std()

sharpe_data = ((rolling_mean - weekly_rf) / rolling_std) * np.sqrt(52)
sharpe_data = sharpe_data.dropna(how='all')

print(f'Sharpe data range: {sharpe_data.shape}')
print(f'Date range: {sharpe_data.index[0].date()} to {sharpe_data.index[-1].date()}')
print(f'\nSample (first 5 stocks, last 5 weeks):')
print(sharpe_data.iloc[-5:, :5])

Sharpe data range: (1044, 267)
Date range: 2006-01-04 to 2025-12-31

Sample (first 5 stocks, last 5 weeks):
Ticker           A   AAPL    ABT    ADBE    ADI
Date                                           
2025-12-03 -0.0083 0.2958 0.2543 -1.5043 0.5756
2025-12-10 -0.1020 0.2253 0.3648 -1.0412 0.5763
2025-12-17 -0.0538 0.0978 0.3282 -0.9179 0.5874
2025-12-24 -0.0515 0.0817 0.2721 -0.9323 0.5483
2025-12-31 -0.1089 0.2162 0.3395 -0.8516 0.5141


In [9]:
# Computing DTW distance matrix for the S&P 500 universe

train_sharpe_data = sharpe_data.loc[TRAIN_START:TRAIN_END]

tickers = train_sharpe_data.columns.tolist()
n = len(tickers)
print(f'Computing DTW distances for {n} stocks ({n*(n-1)//2} pairs), Train period only...')

# dtaidistance requires float64
series = [train_sharpe_data[t].values.astype(np.float64) for t in tickers]

start_time = time.time()
dtw_distance_matrix = dtw.distance_matrix_fast(series)
elapsed = time.time() - start_time

print(f'\nDTW distance matrix computes in {elapsed:.1f} seconds')
print(f'Matrix shape: {dtw_distance_matrix.shape}')

dtw_distance_matrix[dtw_distance_matrix == np.inf] = 0
dtw_distance_matrix = dtw_distance_matrix + dtw_distance_matrix.T

print(f'\nSample of distance matrix (first 5x5):')
print(pd.DataFrame(dtw_distance_matrix[:5, :5], index=tickers[:5], columns=tickers[:5]))

# Locking the result to disk
np.save('dtw_distance_matrix_sp500_train.npy', dtw_distance_matrix)
pd.Series(tickers).to_csv('dtw_tickers_sp500_train.csv', index=False, header=['ticker'])
print('\nSaved dtw_distance_matrix_sp500_train.npy and dtw_tickers_sp500_train.csv')

Computing DTW distances for 267 stocks (35511 pairs), Train period only...

DTW distance matrix computes in 7.3 seconds
Matrix shape: (267, 267)

Sample of distance matrix (first 5x5):
           A    AAPL     ABT    ADBE     ADI
A     0.0000 14.0768 24.8103 14.6522 16.0921
AAPL 14.0768  0.0000 28.0367 24.1182 22.6597
ABT  24.8103 28.0367  0.0000 12.1187 13.2489
ADBE 14.6522 24.1182 12.1187  0.0000 12.9011
ADI  16.0921 22.6597 13.2489 12.9011  0.0000

Saved dtw_distance_matrix_sp500_train.npy and dtw_tickers_sp500_train.csv


In [ ]:
# DTW hierarchical clustering (k=2, 4, 7), Ward linkage
condensed_dtw = squareform(dtw_distance_matrix)
dtw_linkage = linkage(condensed_dtw, method='ward')

dtw_clusters = {}
for k in K_VALUES:
    dtw_clusters[k] = fcluster(dtw_linkage, t=k, criterion='maxclust')
    sizes = pd.Series(dtw_clusters[k]).value_counts().sort_index()
    print(f'k={k}: cluster sizes = {sizes.tolist()}')

# Dendrogram
fig, ax = plt.subplots(figsize=(20,8))
dendrogram(dtw_linkage, labels=tickers, leaf_rotation=90, leaf_font_size=4, ax=ax)
ax.set_title('DTW Hierarchical Clustering Dendrogram - S&P 500 (2005-2010 Train, 20-Year Study)')
ax.set_xlabel('Stock')
ax.set_ylabel('Ward Distance')
plt.tight_layout()
plt.savefig('dtw_dendrogram_sp500_20y.png', dpi=150, bbox_inches='tight')
plt.show()

